# Float RAG Experiment Analysis

This notebook analyzes evaluation reports generated by `python -m app.ai.evals.experiment`.
All logic lives in the `app.ai.evals` package; this notebook only loads report JSONs and visualizes results.

In [ ]:
import glob
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

## 1. Load Experiment Reports

In [ ]:
reports_path = Path("../../evals/reports")
if not reports_path.exists():
    reports_path = Path("evals/reports")

report_files = sorted(glob.glob(str(reports_path / "*.json")))
records = []
for f in report_files:
    with open(f, encoding="utf-8") as fp:
        records.append(json.load(fp))

print(f"Loaded {len(records)} experiment reports.")

## 2. Configuration Comparison with 95% Confidence Intervals

> **Dev Set Notice**: The dev split consists of synthetic template questions (100% contain the gold heading path); not representative of real users. Recall@5 is removed from headline results to avoid synthetic bias; Answer-Bearing Hit@5, MRR, nDCG@5, and Status Accuracy are reported as headline metrics.

In [ ]:
summary_rows = []
for r in records:
    cfg = r.get("config", {})
    ret_over = r.get("retrieval_overall", {})
    ans_over = r.get("answer_overall") or {}
    abh = ret_over.get("answer_bearing_hit") or ret_over.get("recall_at_5", {})
    mrr = ret_over.get("mrr", {})
    ndcg = ret_over.get("ndcg_at_5", {})
    st_acc = ans_over.get("status_accuracy", {})

    summary_rows.append(
        {
            "Run ID": r.get("run_id"),
            "Mode": cfg.get("mode"),
            "Answer-Bearing Hit@5": f"{abh.get('mean', 0):.3f} [{abh.get('ci_95_low', 0):.3f}, {abh.get('ci_95_high', 0):.3f}]",
            "MRR": f"{mrr.get('mean', 0):.3f} [{mrr.get('ci_95_low', 0):.3f}, {mrr.get('ci_95_high', 0):.3f}]",
            "nDCG@5": f"{ndcg.get('mean', 0):.3f} [{ndcg.get('ci_95_low', 0):.3f}, {ndcg.get('ci_95_high', 0):.3f}]",
            "Status Acc": f"{st_acc.get('mean', 0):.3f}",
            "Latency p50": f"{r.get('latency_p50_ms')} ms",
            "Latency p95": f"{r.get('latency_p95_ms')} ms",
        }
    )

df_summary = pd.DataFrame(summary_rows)
df_summary

## 3. Per-Language Retrieval & Answer Metrics

In [ ]:
if records:
    latest = records[-1]
    ret_by_lang = latest.get("retrieval_by_language", {})
    langs = list(ret_by_lang.keys())
    r5_means = [ret_by_lang[lg].get("recall_at_5", {}).get("mean", 0) for lg in langs]
    mrr_means = [ret_by_lang[lg].get("mrr", {}).get("mean", 0) for lg in langs]

    fig, ax = plt.subplots(figsize=(8, 4))
    x = range(len(langs))
    width = 0.35
    ax.bar([i - width / 2 for i in x], r5_means, width, label="Recall@5")
    ax.bar([i + width / 2 for i in x], mrr_means, width, label="MRR")
    ax.set_ylabel("Score")
    ax.set_title(f"Per-Language Retrieval ({latest.get('run_id')})")
    ax.set_xticks(list(x))
    ax.set_xticklabels(langs)
    ax.legend()
    plt.show()

## 4. Abstention Threshold Sweep

In [ ]:
if records:
    sweep = records[-1].get("abstention_threshold_sweep", {})
    if sweep:
        thresholds = [v["threshold"] for v in sweep.values()]
        fa_rates = [v["false_abstain_rate"] for v in sweep.values()]
        fans_rates = [v["false_answer_rate"] for v in sweep.values()]

        plt.figure(figsize=(7, 4))
        plt.plot(thresholds, fa_rates, marker="o", label="False Abstention Rate")
        plt.plot(thresholds, fans_rates, marker="s", label="False Answer Rate")
        plt.xlabel("Min Evidence Score Threshold")
        plt.ylabel("Error Rate")
        plt.title("Abstention Threshold Tradeoff Curve")
        plt.grid(True)
        plt.legend()
        plt.show()

## 5. Error Analysis: Worst Cases Breakdown

In [ ]:
if records:
    worst = records[-1].get("worst_cases", [])
    df_errors = pd.DataFrame(worst)
    if not df_errors.empty:
        print(df_errors[["case_id", "language", "reason_code", "details"]].head(10))
    else:
        print("No error cases recorded in latest run.")